In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [ ]:
from modules.RBF import *
from modules.respy import get_zstack

import numpy as np
from tqdm import tqdm
import tifffile as tf
import cv2 as cv
import matplotlib.pyplot as plt

In [ ]:
# Считываем изображение и нормализуем
data_dir = r'example_data\2025 Mouse_2\2025.05.26 Mouse_2 (DAY 3)\zstack 10 neuron 3'
image = get_zstack(data_dir) / 255.0
print(image.shape)


In [ ]:
# Сжимаем, чтобы точно хватило памяти 
image = np.array([cv.resize(layer, (layer.shape[0]//2, layer.shape[1]//2), interpolation=cv.INTER_CUBIC) for layer in image])
print(image.shape)

In [ ]:
# Задаем параметры сети
sigma = 4.5
NodeX = 64
NodeY = 64
b = 64
epoch = 32
teta = 0.05

In [ ]:
# Считаем внутренние веса сети, на основе выбранных параметров
phi, _ = RBF.construct_phi(image[0], sigma=sigma, NodeX1=NodeX, NodeX2=NodeY, batch_size=b)

In [ ]:
# Проводим обучение сети для каждого слоя и получаем интерполяцию
approx = []
for layer in tqdm(image):
    w = RBF.train_weights(phi, layer, epoch, teta)
    y = RBF.construct_yRBF(w, phi)
    approx.append(y.cpu().numpy())
approx = np.array(approx)

In [ ]:
plt.figure(figsize=(12, 12))
plt.imshow(approx[len(approx)//2])
plt.show()

In [ ]:
# Сохраняем аппроксимацию как восьмитное изображение
if not os.path.exists('temp'):
    os.mkdir('temp')
tf.imwrite(r'temp\rbf_example.tif', np.uint8(np.clip(approx, 0, 1)*255))